In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDOneClassSVM
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    fbeta_score
)
import matplotlib.pyplot as plt
from sklearn.kernel_approximation import RBFSampler
from pyathena.pandas.cursor import PandasCursor
from pyathena import connect
from itertools import product
import csv

In [53]:
ACCESS_KEY = ""
SECRET_KEY = ""

In [54]:
cursor = connect(
    s3_staging_dir="s3://bdp-athena-results/",
    region_name="eu-north-1",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    cursor_class=PandasCursor,
).cursor()

features_table_name = "bdp.scaled_features"

In [2]:
X_train = pd.read_parquet("data").to_numpy(dtype=float)
validation_data = pd.read_parquet("benchmark")

In [3]:
X_val = validation_data.drop(columns=['label']).to_numpy(dtype=float)
y_val = validation_data['label'].values

In [4]:
y_val = np.where(y_val, -1, 1)

In [5]:
class VotingSGDOneClassSVM:
    def __init__(self, estimators):
        self.estimators = estimators

    def fit(self, X):        
        for label, model in self.estimators:
            print(f"Training for {label}")            
            model.fit(X)

    def predict(self, X):
        predictions = np.array([model.predict(X) for _, model in self.estimators])
        votes = np.sign(np.sum(predictions, axis=0))
        return votes

In [13]:
gamma_values = np.arange(0.1, 10.0, 0.1).tolist()

param_values = {"gamma": gamma_values, "nu": [0.01, 0.05, 0.1, 0.2], "tol": [1e-7], "eta0": [1e-6]}

keys = param_values.keys()
values = param_values.values()
combinations = [dict(zip(keys, combo)) for combo in product(*values)]

In [16]:
csv_file = "svm_results_tuning.csv"

with open(csv_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["gamma", "nu", "tol", "eta0", "Precision", "Recall Normal", "Recall Anomaly", "F1-Score", "Accuracy", "AUC", "F2-Score"])

    for i, combo in enumerate(combinations, start=1):
        nystroem = RBFSampler(
            gamma=combo["gamma"], 
            n_components=1000,
            random_state=42
        )

        sgd_ocsvm = SGDOneClassSVM(
            nu=combo["nu"],
            shuffle=True,
            tol=combo["tol"],
            random_state=42,
            eta0=combo["eta0"],
            max_iter=10000
        )

        X_batch_transformed = nystroem.fit_transform(X_val)

        sgd_ocsvm.fit(X_batch_transformed)

        X_val_transformed = nystroem.transform(X_val)
        y_pred = sgd_ocsvm.predict(X_val_transformed)

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, pos_label=1)
        recall_normal = recall_score(y_val, y_pred, pos_label=1)
        recall_anomaly = recall_score(y_val, y_pred, pos_label=-1)
        f1 = f1_score(y_val, y_pred, pos_label=1)
        auc = roc_auc_score(y_val, y_pred)
        f2 = fbeta_score(y_val, y_pred, beta=2, pos_label=1)

        print(i)

        writer.writerow([
                combo["gamma"], combo["nu"], combo["tol"], combo["eta0"],
                precision, recall_normal, recall_anomaly, f1, accuracy, auc, f2
        ])

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
